# ERCOT Electricity Market Data Cleaning (v2)

This notebook cleans and aggregates **ERCOT load, settlement point price, and generation-by-fuel** datasets into **Power BI–ready CSVs**.

### What’s new in v2
- Adds **seasonality features** (`season`, `season_year`) for Load / Price / Generation
- Produces **seasonal yearly** rollups for Power BI (trend by season across years)
- More robust **generation workbook parsing** (detect header row, handle month sheets)

### Output (for Power BI)
You will export clean datasets:
- Hourly Load (date/hour)
- Monthly Load (avg)
- Monthly Prices (avg + volatility)
- Monthly Generation (sum + renewable share)
- Seasonal-Yearly: Load / Price / Generation

## Data inputs (what you need to upload)

Upload **ERCOT Excel files (.xlsx)** into this Colab session. The notebook expects these patterns:

### Load
- Files containing: `Native_Load`  
  Example: `Native_Load_2023.xlsx`

### Prices
- Files containing: `RPT` (case-insensitive)  
  Example: `RPT_SPP_2023.xlsx`

### Generation
- Files containing: `IntGenByFuel` (case-insensitive)  
  Example: `IntGenByFuel_2023.xlsx`

> Tip: Keep filenames consistent by year (makes debugging easier).

## Environment setup

This notebook is designed for **Google Colab** and uses:
- `pandas`, `numpy`
- `google.colab.files` for upload/download

If you run locally, replace `files.upload()` / `files.download()` with local file paths.

In [ ]:
import os, calendar
import pandas as pd
import numpy as np
from google.colab import files




## Helper: Season features

We add:
- `year` and `month` extracted from a date column
- `season` mapped using meteorological seasons:
  - Winter = Dec/Jan/Feb
  - Spring = Mar/Apr/May
  - Summer = Jun/Jul/Aug
  - Fall = Sep/Oct/Nov
- `season_year` fixes the winter-year edge case:
  - **Dec 2023 belongs to Winter 2024**
  - Jan/Feb 2024 also belong to Winter 2024

This makes seasonal charts consistent in Power BI.

In [ ]:
def add_season_cols(df, date_col):
    d = pd.to_datetime(df[date_col], errors="coerce")
    out = df.copy()

    out["year"] = d.dt.year
    out["month"] = d.dt.month

    out["season"] = np.select(
        [
            out["month"].isin([12, 1, 2]),
            out["month"].isin([3, 4, 5]),
            out["month"].isin([6, 7, 8]),
            out["month"].isin([9, 10, 11]),
        ],
        ["Winter", "Spring", "Summer", "Fall"],
        default="Unknown"   # ← FIX: string default
    ).astype("object")

    # Winter spans years: Dec belongs to next year's winter
    out["season_year"] = np.where(
        out["month"] == 12,
        out["year"] + 1,
        out["year"]
    )

    return out


## Step 1 — Upload ERCOT Excel files

Upload all `.xlsx` files needed for:
- Native load
- Settlement point prices
- Generation by fuel

Colab will store them in `/content/`.

In [ ]:
uploaded = files.upload()



## Step 2 — Load data (Native Load)

### Goal
Create a clean **hourly load** dataset and aggregate it to:
- **Monthly average load**
- **Seasonal-yearly average load** (v2)

### Key cleaning rules
- Parse `Hour Ending` safely (`errors="coerce"`)
- Drop rows where `Hour Ending` is missing
- Extract:
  - `date` (day)
  - `hour` (0–23)
- Convert `ERCOT` to numeric as `load_mw`

### Load outputs
- `load_hourly`: columns = `date`, `hour`, `load_mw`
- `load_monthly`: monthly mean load (`avg_load_mw`)
- `load_seasonal_yearly` (v2): seasonal mean load by `season_year` + `season`

In [ ]:
xlsx_files = [f for f in os.listdir("/content") if f.endswith(".xlsx")]
xlsx_files


In [ ]:
load_files = [f for f in xlsx_files if "Native_Load" in f]

dfs = [pd.read_excel(f"/content/{f}") for f in load_files]
load_raw = pd.concat(dfs, ignore_index=True)
load_raw.head()


In [ ]:
load_raw["Hour Ending"] = pd.to_datetime(load_raw["Hour Ending"], errors="coerce")
load_raw = load_raw.dropna(subset=["Hour Ending"])

load_raw["date"] = load_raw["Hour Ending"].dt.date
load_raw["hour"] = load_raw["Hour Ending"].dt.hour


In [ ]:
load_hourly = load_raw[["date", "hour", "ERCOT"]].rename(
    columns={"ERCOT": "load_mw"}
)

load_hourly["load_mw"] = pd.to_numeric(load_hourly["load_mw"], errors="coerce")
load_hourly = load_hourly.dropna(subset=["load_mw"])
load_hourly.head()


In [ ]:
load_hourly["date"] = pd.to_datetime(load_hourly["date"])

load_monthly = (
    load_hourly
    .groupby(pd.Grouper(key="date", freq="M"))["load_mw"]
    .mean()
    .reset_index()
    .rename(columns={"load_mw": "avg_load_mw"})
)
load_monthly.head()


In [ ]:
load_hourly_s = add_season_cols(load_hourly, "date")

# Seasonal per year (avg load)
load_seasonal_yearly = (
    load_hourly_s
    .groupby(["season_year", "season"], as_index=False)["load_mw"]
    .mean()
    .rename(columns={"load_mw": "avg_load_mw"})
)

# Seasonal pattern across years is shown by plotting:
# X=season_year, Legend=season, Y=avg_load_mw
load_seasonal_yearly.head()


## Step 3 — Price data (Settlement Point Prices)

### Goal
Clean ERCOT settlement point price data and aggregate into:
- **Monthly average price**
- **Monthly volatility** (standard deviation)
- **Seasonal-yearly price trends** (v2)

### Notes on time fields
ERCOT price files often contain:
- `Delivery Date`
- `Delivery Hour` (1–24) → we convert to 0–23
- `Delivery Interval` (1–4) → mapped to minutes: 0/15/30/45

### Price outputs
- `prices_monthly`: columns = `datetime`, `avg_price`, `price_volatility`
- `prices_seasonal_yearly` (v2):
  - avg seasonal price
  - seasonal volatility
  - number of hourly observations (`n_hours`)

In [ ]:
price_files = [f for f in xlsx_files if "RPT" in f.upper()]
print("Price files:", price_files)

price_dfs = []

for f in price_files:
    print(f"Processing file: {f}")
    path = f"/content/{f}"
    xls = pd.ExcelFile(path)

    for sheet in xls.sheet_names:
        if sheet.lower() in ["disclaimer", "readme"]:
            continue

        print(f"  → Sheet: {sheet}")
        try:
            df = pd.read_excel(
                path,
                sheet_name=sheet,
                usecols=[
                    "Delivery Date",
                    "Delivery Hour",
                    "Delivery Interval",
                    "Settlement Point Name",
                    "Settlement Point Price"
                ]
            )
            price_dfs.append(df)
        except Exception:
            continue

prices_raw = pd.concat(price_dfs, ignore_index=True)
prices_raw.head()


In [ ]:
prices_raw["Delivery Date"] = pd.to_datetime(prices_raw["Delivery Date"], errors="coerce")
prices_raw["Delivery Hour"] = pd.to_numeric(prices_raw["Delivery Hour"], errors="coerce")
prices_raw["Delivery Interval"] = pd.to_numeric(prices_raw["Delivery Interval"], errors="coerce")

prices_raw = prices_raw.dropna(subset=["Delivery Date", "Delivery Hour", "Delivery Interval"])

prices_raw["hour"] = (prices_raw["Delivery Hour"] - 1).astype(int)
prices_raw["minute"] = ((prices_raw["Delivery Interval"] - 1) * 15).astype(int)

prices_raw["datetime"] = (
    prices_raw["Delivery Date"]
    + pd.to_timedelta(prices_raw["hour"], unit="h")
    + pd.to_timedelta(prices_raw["minute"], unit="m")
)


In [ ]:
prices_raw["price"] = pd.to_numeric(
    prices_raw["Settlement Point Price"], errors="coerce"
)
prices_raw = prices_raw.dropna(subset=["price"])

prices_monthly = (
    prices_raw
    .groupby(pd.Grouper(key="datetime", freq="M"))["price"]
    .agg(avg_price="mean", price_volatility="std")
    .reset_index()
)
prices_monthly.head()


In [ ]:
# Build hourly average price first (reduces noise and size)
prices_raw["date"] = prices_raw["datetime"].dt.floor("D")

prices_hourly = (
    prices_raw
    .groupby(["date", "hour"], as_index=False)["price"]
    .mean()
    .rename(columns={"price": "avg_price"})
)

prices_hourly_s = add_season_cols(prices_hourly, "date")

# Seasonal per year: avg price + volatility (std) + n_hours
prices_seasonal_yearly = (
    prices_hourly_s
    .groupby(["season_year", "season"], as_index=False)
    .agg(
        avg_price=("avg_price", "mean"),
        price_volatility=("avg_price", "std"),
        n_hours=("avg_price", "count")
    )
)

prices_seasonal_yearly.head()


## Step 4 — Generation by fuel (IntGenByFuel)

### Goal
Load all `IntGenByFuel*.xlsx` workbooks, extract daily generation totals by fuel, then create:
- **Daily wide table** (columns = fuel types)
- **Monthly totals**
- **Renewable share (% of total)**
- **Seasonal-yearly totals** (v2)

### Why this part is “robust”
ERCOT generation files sometimes have headers that don’t start on row 0.
This notebook:
- detects the header row by scanning for keywords (Date / Fuel / Total)
- reads only month sheets (Jan–Dec)
- skips sheets that don’t match expected structure

### Renewable definition used
A fuel column is treated as renewable if its name contains:
- `wind`, `solar`, `hydro`, or `biomass`

Renewable metrics produced:
- `renewable_gen`
- `renewable_pct = renewable_gen / total_gen`

In [ ]:
# ---- Find generation files ----
xlsx_files = [f for f in os.listdir("/content") if f.lower().endswith(".xlsx")]
gen_files = sorted([f for f in xlsx_files if "intgenbyfuel" in f.lower()])

if not gen_files:
    raise FileNotFoundError("❌ No IntGenbyFuel*.xlsx found in /content. Upload your IntGenbyFuel files first.")

print("✅ Found generation files:", gen_files)

month_names = list(calendar.month_abbr)[1:]  # Jan..Dec

def detect_header_row(path, sheet_name, search_rows=40):
    """Find the row index that contains header names like Date/Fuel/Total."""
    tmp = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=search_rows)
    for r in range(search_rows):
        row = tmp.iloc[r].tolist()
        vals = [str(x).strip().lower() for x in row if pd.notna(x)]
        if not vals:
            continue
        has_date = any(("date" == v) or ("date" in v) for v in vals)
        has_fuel = any(("fuel" == v) or ("fuel" in v) for v in vals)
        has_total = any(("total" == v) or ("total" in v) for v in vals)
        if has_date and has_fuel and has_total:
            return r
    return None

def standardize_cols(cols):
    return [str(c).strip().lower() for c in cols]

def pick_col_index(cols_lower, keys):
    for i, c in enumerate(cols_lower):
        for k in keys:
            if c == k or k in c:
                return i
    return None

def read_one_generation_file(path):
    xls = pd.ExcelFile(path)
    sheets = [s for s in xls.sheet_names if s in month_names]  # only months

    out_frames = []
    for sh in sheets:
        hdr = detect_header_row(path, sh)
        if hdr is None:
            # fallback guesses (some files place headers at row 0..8)
            for guess in range(0, 10):
                try:
                    df_try = pd.read_excel(path, sheet_name=sh, header=guess, nrows=5)
                    cols_lower = standardize_cols(df_try.columns)
                    if any("date" in c for c in cols_lower) and any("fuel" in c for c in cols_lower) and any("total" in c for c in cols_lower):
                        hdr = guess
                        break
                except Exception:
                    continue

        if hdr is None:
            print(f"⚠️ Skipping {os.path.basename(path)} | {sh}: could not detect header row.")
            continue

        df = pd.read_excel(path, sheet_name=sh, header=hdr)
        cols_lower = standardize_cols(df.columns)

        date_i = pick_col_index(cols_lower, ["date"])
        fuel_i = pick_col_index(cols_lower, ["fuel"])
        total_i = pick_col_index(cols_lower, ["total"])

        if date_i is None or fuel_i is None or total_i is None:
            print(f"⚠️ Skipping {os.path.basename(path)} | {sh}: missing Date/Fuel/Total. Columns: {list(df.columns)}")
            continue

        sub = df.iloc[:, [date_i, fuel_i, total_i]].copy()
        sub.columns = ["date", "fuel", "total"]
        sub["source_file"] = os.path.basename(path)
        sub["sheet"] = sh
        out_frames.append(sub)

    if not out_frames:
        return pd.DataFrame(columns=["date", "fuel", "total", "source_file", "sheet"])

    return pd.concat(out_frames, ignore_index=True)

# ---- Read all years ----
gen_raw = pd.concat(
    [read_one_generation_file(f"/content/{f}") for f in gen_files],
    ignore_index=True
)

if gen_raw.empty:
    raise ValueError("❌ No generation rows loaded. Your sheets may use different names—inspect one 'Jan' sheet manually.")

# ---- Clean types + missing values ----
gen_raw["date"] = pd.to_datetime(gen_raw["date"], errors="coerce")
gen_raw["fuel"] = gen_raw["fuel"].astype(str).str.strip()
gen_raw["total"] = pd.to_numeric(gen_raw["total"], errors="coerce").fillna(0)

gen_raw = gen_raw.dropna(subset=["date", "fuel"]).copy()

print("✅ gen_raw preview:")
display(gen_raw.head())

# ---- Daily totals by fuel ----
gen_daily = (
    gen_raw.groupby(["date", "fuel"], as_index=False)["total"]
    .sum()
    .rename(columns={"total": "gen_total"})
)

# ---- Wide daily table (columns = fuels) ----
gen_daily_wide = gen_daily.pivot_table(
    index="date",
    columns="fuel",
    values="gen_total",
    aggfunc="sum",
    fill_value=0
).reset_index()

# ---- Renewable % (robust keyword match) ----
fuel_cols = [c for c in gen_daily_wide.columns if c != "date"]
renew_cols = [c for c in fuel_cols if any(k in str(c).lower() for k in ["wind", "solar", "hydro", "biomass"])]

gen_daily_wide["total_gen"] = gen_daily_wide[fuel_cols].sum(axis=1)
gen_daily_wide["renewable_gen"] = gen_daily_wide[renew_cols].sum(axis=1) if renew_cols else 0
gen_daily_wide["renewable_pct"] = np.where(
    gen_daily_wide["total_gen"] > 0,
    gen_daily_wide["renewable_gen"] / gen_daily_wide["total_gen"],
    np.nan
)

gen_daily_wide["month"] = gen_daily_wide["date"].dt.to_period("M").astype(str)

# ---- Monthly aggregation ----
gen_monthly = (
    gen_daily_wide.groupby("month", as_index=False)
    .agg({**{c: "sum" for c in fuel_cols}, "total_gen": "sum", "renewable_gen": "sum"})
)
gen_monthly["renewable_pct"] = np.where(
    gen_monthly["total_gen"] > 0,
    gen_monthly["renewable_gen"] / gen_monthly["total_gen"],
    np.nan
)

print("✅ generation_daily preview:")
display(gen_daily_wide.head())
print("✅ generation_monthly preview:")
display(gen_monthly.head())

# ---- Export + download ----
daily_path = "/content/generation_daily_clean.csv"
monthly_path = "/content/generation_monthly_clean.csv"

gen_daily_wide.to_csv(daily_path, index=False)
gen_monthly.to_csv(monthly_path, index=False)

print("✅ Saved:", daily_path)
print("✅ Saved:", monthly_path)

files.download(daily_path)
files.download(monthly_path)


## Seasonality (v2 feature)

This notebook adds **seasonality features** to Load, Price, and Generation datasets.

### Columns added
- `season`: Winter / Spring / Summer / Fall (meteorological seasons)
- `season_year`: fixes Winter spanning two calendar years  
  - December is assigned to the **next year’s winter**
  - Example: Dec 2023 → Winter **season_year = 2024**

### Why this matters
It enables consistent seasonal trend charts in Power BI:
- X-axis: `season_year`
- Legend: `season`
- Y-axis: avg load / avg price / renewable %

In [ ]:
gen_daily_wide_s = add_season_cols(gen_daily_wide, "date")

exclude = {"date", "year", "month", "season", "season_year"}
fuel_cols_s = [c for c in gen_daily_wide_s.columns if c not in exclude and pd.api.types.is_numeric_dtype(gen_daily_wide_s[c])]

renew_cols_s = [c for c in fuel_cols_s if any(k in c.lower() for k in ["wind", "solar", "hydro", "biomass"])]

gen_seasonal_yearly = (
    gen_daily_wide_s
    .groupby(["season_year", "season"], as_index=False)[fuel_cols_s]
    .sum()
)

gen_seasonal_yearly["total_gen"] = gen_seasonal_yearly[fuel_cols_s].sum(axis=1)
gen_seasonal_yearly["renewable_gen"] = gen_seasonal_yearly[renew_cols_s].sum(axis=1) if renew_cols_s else 0
gen_seasonal_yearly["renewable_pct"] = np.where(
    gen_seasonal_yearly["total_gen"] > 0,
    gen_seasonal_yearly["renewable_gen"] / gen_seasonal_yearly["total_gen"],
    np.nan
)

gen_seasonal_yearly.head()


In [ ]:
gen_raw["date"] = pd.to_datetime(gen_raw["date"], errors="coerce")
gen_raw["total"] = pd.to_numeric(gen_raw["total"], errors="coerce").fillna(0)



In [ ]:
import numpy as np

# Ensure month column exists
gen_daily_wide["month"] = gen_daily_wide["date"].dt.to_period("M").astype(str)

# Select ONLY numeric columns (fuel columns + numeric metrics)
numeric_cols = gen_daily_wide.select_dtypes(include="number").columns.tolist()

# Monthly aggregation (safe: no datetime summation)
gen_monthly = (
    gen_daily_wide
    .groupby("month", as_index=False)[numeric_cols]
    .sum()
)

# OPTIONAL: recompute renewable percentage if components exist
# (robust to different fuel naming)
fuel_cols = [c for c in gen_monthly.columns if c not in ["month"]]

renew_cols = [
    c for c in fuel_cols
    if any(k in c.lower() for k in ["wind", "solar", "hydro", "biomass"])
]

if renew_cols:
    gen_monthly["total_gen"] = gen_monthly[fuel_cols].sum(axis=1)
    gen_monthly["renewable_gen"] = gen_monthly[renew_cols].sum(axis=1)
    gen_monthly["renewable_pct"] = np.where(
        gen_monthly["total_gen"] > 0,
        gen_monthly["renewable_gen"] / gen_monthly["total_gen"],
        np.nan
    )

# Preview
gen_monthly.head()


In [ ]:
import numpy as np

# Identify numeric fuel columns (everything except the month key)
fuel_cols = [c for c in gen_monthly.columns if c != "month"]

# Identify renewable columns by keyword (robust to naming differences)
renew_cols = [
    c for c in fuel_cols
    if any(k in c.lower() for k in ["wind", "solar", "hydro", "biomass"])
]

# Total generation (numeric only)
gen_monthly["total_gen"] = gen_monthly[fuel_cols].sum(axis=1)

# Renewable generation
gen_monthly["renewable_gen"] = (
    gen_monthly[renew_cols].sum(axis=1) if renew_cols else 0
)

# Renewable percentage (safe divide)
gen_monthly["renewable_pct"] = np.where(
    gen_monthly["total_gen"] > 0,
    gen_monthly["renewable_gen"] / gen_monthly["total_gen"],
    np.nan
)

# Preview
gen_monthly.head()


## Step 5 — Export (Power BI ready)

This notebook exports clean CSV files for dashboards and analysis.

### Final exports
- `load_hourly_clean.csv`
- `load_monthly_clean.csv`
- `prices_monthly_clean.csv`
- `generation_monthly_clean.csv`
- `load_seasonal_yearly_clean.csv` (v2)
- `prices_seasonal_yearly_clean.csv` (v2)
- `generation_seasonal_yearly_clean.csv` (v2)

> Recommendation: Store these in a `/data/processed/` folder in GitHub (but keep raw ERCOT files out of the repo).

## Quick sanity checks

Before exporting, we print dataset shapes to confirm:
- data loaded successfully
- aggregations produced non-empty outputs
- generation/monthly summaries look reasonable

In [ ]:
# Quick sanity check (optional but recommended)
print("load_hourly:", load_hourly.shape)
print("load_monthly:", load_monthly.shape)
print("prices_monthly:", prices_monthly.shape)
print("gen_monthly:", gen_monthly.shape)

# Export to CSV (Power BI ready)
load_hourly.to_csv("load_hourly_clean.csv", index=False)
load_monthly.to_csv("load_monthly_clean.csv", index=False)
prices_monthly.to_csv("prices_monthly_clean.csv", index=False)
gen_monthly.to_csv("generation_monthly_clean.csv", index=False)
load_seasonal_yearly.to_csv("load_seasonal_yearly_clean.csv", index=False)
prices_seasonal_yearly.to_csv("prices_seasonal_yearly_clean.csv", index=False)
gen_seasonal_yearly.to_csv("generation_seasonal_yearly_clean.csv", index=False)

# Download files
files.download("load_hourly_clean.csv")
files.download("load_monthly_clean.csv")
files.download("prices_monthly_clean.csv")
files.download("generation_monthly_clean.csv")
files.download("load_seasonal_yearly_clean.csv")
files.download("prices_seasonal_yearly_clean.csv")
files.download("generation_seasonal_yearly_clean.csv")



## Troubleshooting

### If generation loads empty
- Confirm your file contains month sheets named exactly like: `Jan`, `Feb`, ..., `Dec`
- Open one sheet and verify it contains columns similar to: Date / Fuel / Total
- Some ERCOT files may rename these columns slightly; update the keyword match if needed

### If price files skip many sheets
- Some sheets may not contain the expected columns
- The code intentionally ignores `Disclaimer` / `Readme`
- Inspect `xls.sheet_names` and adjust `usecols` if ERCOT changes formats